In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
import pint
import pint_xarray

In [ ]:
seapopym_v1 = xr.open_zarr(f"/zooplankton_no_transport_seapopym_v2.zarr/")
seapopym_v1 = seapopym_v1["biomass"]
seapopym_v1 = seapopym_v1.sel(time=slice("2004-01-01", "2004-12-31")).load()
seapopym_v1 = seapopym_v1.assign_coords(x=(seapopym_v1.x % 360)).sortby("x")
seapopym_v1

In [ ]:
seapopym_v2 = xr.open_zarr(f"./zooplankton_transport_pacific_seapopym_v2.zarr")
# seapopym_v2 = xr.open_zarr(f"./zooplankton_transport_pacific_biomass_only.zarr")
# seapopym_v2 = xr.open_zarr("./zooplankton_numba_transport_pacific_seapopym_v2.zarr")
seapopym_v2 = seapopym_v2["biomass"]
seapopym_v2 = seapopym_v2.sel(time=slice("2004-01-01", "2004-12-31")).load()
seapopym_v2 = seapopym_v2.resample(time="1D").mean()
seapopym_v2

In [ ]:
# seapodym_lmtl = xr.open_zarr("./pacific_seapodym/result.zarr")
seapodym_lmtl = xr.open_zarr("./zooplankton_numba_transport_pacific_seapopym_v2.zarr")
seapodym_lmtl = seapodym_lmtl["biomass"]
seapodym_lmtl = seapodym_lmtl.resample({"time": "1D"}).mean()
seapodym_lmtl = seapodym_lmtl.sel(time=slice("2004-01-01", "2004-12-31")).load()
seapodym_lmtl

In [ ]:
temperature = xr.open_zarr("../pacific_lmtl_forcings.zarr/")
temperature = temperature["temperature"]
temperature = temperature.sel(time=slice("2004-01-01", "2004-12-31")).load()
temperature

In [ ]:
seapopym_v1 = seapopym_v1.reindex_like(seapopym_v2)
seapodym_lmtl = seapodym_lmtl.reindex_like(seapopym_v2)
temperature = temperature.reindex_like(seapopym_v2)

In [ ]:
bias_v1 = seapodym_lmtl - seapopym_v1
bias_v2 = seapodym_lmtl - seapopym_v2
mpe_v1 = (bias_v1) / seapodym_lmtl
mpe_v2 = (bias_v2) / seapodym_lmtl

# bias_v1 = seapodym_lmtl - seapopym_v1
# bias_v2 = seapopym_v2 - seapopym_v1
# mpe_v1 = (bias_v1) / seapodym_lmtl
# mpe_v2 = (bias_v2) / seapopym_v2

In [ ]:
vmax = 0.5
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

bias_v1.mean("time").plot(vmax=vmax, ax=axes[0])
axes[0].set_title("Bias v1")

bias_v2.mean("time").plot(vmax=vmax, ax=axes[1])
axes[1].set_title("Bias v2")

plt.show()

In [ ]:
vmax = 0.5

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

mpe_v1.mean("time").plot(vmax=vmax, ax=axes[0])
axes[0].set_title("MPE v1")

mpe_v2.mean("time").plot(vmax=vmax, ax=axes[1])
axes[1].set_title("MPE v2")

plt.show()

In [ ]:
min_max = 1
data = pd.DataFrame(
    {
        "mpe_v1": mpe_v1.resample({"time": "1MS"}).mean().data.flatten(),
        "mpe_v2": mpe_v2.resample({"time": "1MS"}).mean().data.flatten(),
        "temperature": temperature.resample({"time": "1MS"}).mean().data.flatten(),
    }
)
data = data[(data["mpe_v1"] < min_max) & (data["mpe_v1"] > -min_max)]
data = data[(data["mpe_v2"] < min_max) & (data["mpe_v2"] > -min_max)]
data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
vmax = 170
vmin = 20
sns.histplot(
    data=data,
    x="temperature",
    y="mpe_v1",
    cbar=True,
    vmin=vmin,
    vmax=vmax,
    ax=axes[0],
    cmap="viridis",
)
axes[0].set_title("MPE between SeapoPym no transport and LMTL")
axes[0].hlines(0, xmin=0, xmax=30, color="red")
axes[0].set_xlim((10, 30))

sns.histplot(
    data=data,
    x="temperature",
    y="mpe_v2",
    cbar=True,
    vmin=vmin,
    vmax=vmax,
    ax=axes[1],
    cmap="viridis",
)
# axes[1].set_title("MPE between SeapoPym transport and LMTL")
axes[1].set_title("MPE between SeapoPym no transport and SeapoPym transport")
axes[1].hlines(0, xmin=0, xmax=30, color="red")
axes[1].set_xlim((10, 30))

plt.show()